In [ ]:
import sys
!{sys.executable} -m pip install python-dotenv

In [ ]:
"""
Test Job.get_next_request() integration with LLM_API.

This script verifies that the chat_history format returned by Job.get_next_request()
is compatible with LLM_API.process_request() method. It can be converted to a 
Jupyter notebook for interactive testing.
"""

import dbzero as db0
from collections import namedtuple
from statek.agent import Agent
from statek.executors.job import Job, JobDef, JobStatus
from statek.executors.chat_log_item import ChatLogItem
from statek.pyenv import PyEnv
from statek.llm_api import LLM_API, OpenRouter_API
from statek.settings import get_provider_settings
from statek.executors.utils import run_jobs_loop


In [ ]:
# Initialize dbzero
db0.init(".dbzero_data")
db0.open("test-prefix-roon-jobs-loop")

In [ ]:
# sample tools mocs

def add(a: int, b: int) -> int:
    """Adds two elements"""
    return a + b

def multiply(a: int, b: int) -> int:
    """multiply two elements"""
    return a * b

def exit(reason: str):
    print(reason)

In [ ]:
"""Create a Job instance"""
# Create agent and pyenv
agent = Agent(_system_prompt="""You are a helpful assistant. You solves given equations, 
but can only calculate one thin at a time. Need to use provided tools: \n{tools} \nYou only return one tool call at time e. g sum(2,5). Send only operation wrapped in print().
No comments just for eg: print(add(1,2)).""", _tools=[add,multiply])
pyenv = PyEnv(local_state={
    "multiply":multiply,
    "add":add
})

max_jobs = 50
def start_jobs(capacity: int):
    global max_jobs
    print(f"Run start jobs for :{capacity} -- {max_jobs}")
    if max_jobs <=0:
        print(f"Finished creating jobs")
    capacity = min(capacity, max_jobs)
    print(f"Creating jobs: {capacity}")
    for i in range(capacity):
        print(f"Creating {i} job")
        # Create job definition and job
        job_def = JobDef(
            agent=agent,
            description="Solve this problem: {goal}",
            goal="2 + 3 * 5 + 4 * 2",
            warmup_code=None
        )
        job = Job(
            job_def=job_def,
            model_family="test",
            model="openai/gpt-5",
            job_status=JobStatus.READY,
            py_env=pyenv
        )
        max_jobs -= 1


### Test 1: Check if run_jobs_loop can create and execute job

In [ ]:
from dotenv import load_dotenv
load_dotenv("./.env")


In [ ]:
result = await run_jobs_loop(25, "OPENROUTER", start_jobs)